**ワンホットエンコーディング(ダミー変数化)**

In [21]:
import pandas as pd
import numpy as np
import mglearn
from sklearn.model_selection import train_test_split
import os

In [9]:
#このファイルにはコラム名を含んだヘッダがないので、header=Noneを指定して読み込む
adult_path = os.path.join(mglearn.datasets.DATA_PATH, "adult.data")
data = pd.read_csv(
    adult_path,
    header=None,
    index_col=False,
    names = [
        "age", "workclass", "fnlwgt", "education", "education-num",
        "marital-status", "occupation", "relationship", "race", "gender",
        "capital-gain", "capital-loss", "hours-per-week", "native-country",
        "income"]
)

#解説のために、いくつかのカラムだけを選択
data = data[
    ["age","workclass","education","gender","hours-per-week","occupation","income"]
]

display(data.head())

,age,workclass,education,gender,hours-per-week,occupation,income
0,39,State-gov,Bachelors,Male,40,Adm-clerical,<=50K
1,50,Self-emp-not-inc,Bachelors,Male,13,Exec-managerial,<=50K
2,38,Private,HS-grad,Male,40,Handlers-cleaners,<=50K
3,53,Private,11th,Male,40,Handlers-cleaners,<=50K
4,28,Private,Bachelors,Female,40,Prof-specialty,<=50K


In [13]:
#各行二ぐ組まれているユニークな値とその頻度を表示する
data["gender"].value_counts()

gender
Male      21790
Female    10771
Name: count, dtype: int64

訓練データとテストデータの双方が入ったデータフレームに対してget_dummiesを呼ぶか、訓練データとテストデータに対してそれぞれget_dummiesを読んだ後に同じコラム名を含んでいるかを確認して、両方が同じ意味にようにしなければならない

In [16]:
#ワンホットエンコーディングを行う
print("Original features:\n",list(data.columns),"\n")
data_dummies = pd.get_dummies(data)
print("Features after get_dummies:\n",list(data_dummies.columns))

Original features:
 ['age', 'workclass', 'education', 'gender', 'hours-per-week', 'occupation', 'income'] 

Features after get_dummies:
 ['age', 'hours-per-week', 'workclass_ ?', 'workclass_ Federal-gov', 'workclass_ Local-gov', 'workclass_ Never-worked', 'workclass_ Private', 'workclass_ Self-emp-inc', 'workclass_ Self-emp-not-inc', 'workclass_ State-gov', 'workclass_ Without-pay', 'education_ 10th', 'education_ 11th', 'education_ 12th', 'education_ 1st-4th', 'education_ 5th-6th', 'education_ 7th-8th', 'education_ 9th', 'education_ Assoc-acdm', 'education_ Assoc-voc', 'education_ Bachelors', 'education_ Doctorate', 'education_ HS-grad', 'education_ Masters', 'education_ Preschool', 'education_ Prof-school', 'education_ Some-college', 'gender_ Female', 'gender_ Male', 'occupation_ ?', 'occupation_ Adm-clerical', 'occupation_ Armed-Forces', 'occupation_ Craft-repair', 'occupation_ Exec-managerial', 'occupation_ Farming-fishing', 'occupation_ Handlers-cleaners', 'occupation_ Machine-op-i

In [17]:
data_dummies.head()

,age,hours-per-week,workclass_ ?,workclass_ Federal-gov,workclass_ Local-gov,workclass_ Never-worked,workclass_ Private,workclass_ Self-emp-inc,workclass_ Self-emp-not-inc,workclass_ State-gov,...,occupation_ Machine-op-inspct,occupation_ Other-service,occupation_ Priv-house-serv,occupation_ Prof-specialty,occupation_ Protective-serv,occupation_ Sales,occupation_ Tech-support,occupation_ Transport-moving,income_ <=50K,income_ >50K
0,39,40,False,False,False,False,False,False,False,True,...,False,False,False,False,False,False,False,False,True,False
1,50,13,False,False,False,False,False,False,True,False,...,False,False,False,False,False,False,False,False,True,False
2,38,40,False,False,False,False,True,False,False,False,...,False,False,False,False,False,False,False,False,True,False
3,53,40,False,False,False,False,True,False,False,False,...,False,False,False,False,False,False,False,False,True,False
4,28,40,False,False,False,False,True,False,False,False,...,False,False,False,True,False,False,False,False,True,False


In [ ]:
#モデルを学習させる前にターゲット変数を分離

#DataFrame型
features = data_dummies.loc[:,"age":"occupation_ Transport-moving"]

#Numpy配列を取り出す
x = features.to_numpy()
y = data_dummies["income_ >50K"].to_numpy()

#これでデータはscikit-learnで使える形になった
print("x shape:{}".format(x.shape))
print("y shape:{}".format(y.shape))

x shape:(32561, 44)
y shape:(32561,)


In [23]:
from sklearn.linear_model import LogisticRegression

x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    random_state=0
)

logreg = LogisticRegression(max_iter=1000)
logreg.fit(x_train,y_train)

print("Test score:{:.2f}".format(logreg.score(x_test,y_test)))

Test score:0.81


**数値でエンコードされているカテゴリ**

In [26]:
#整数特徴量とカテゴリ文字特徴量からなるデータフレームを作る
demo_df = pd.DataFrame({
    "Integer Feature":[0,1,2,1],
     "Categorical Feature":["socks","fox","socks","box"]
})

display(demo_df)

,Integer Feature,Categorical Feature
0,0,socks
1,1,fox
2,2,socks
3,1,box


In [27]:
#get_dummiesを実行しても、整数特徴量は変化しない
pd.get_dummies(demo_df)

,Integer Feature,Categorical Feature_box,Categorical Feature_fox,Categorical Feature_socks
0,0,False,False,True
1,1,False,True,False
2,2,False,False,True
3,1,True,False,False


In [ ]:
#columns引数を使って明示的に指定する
demo_df["Integer Feature"] = demo_df["Integer Feature"].astype(str) #必須ではない
pd.get_dummies(demo_df,columns=["Integer Feature","Categorical Feature"])

,Integer Feature_0,Integer Feature_1,Integer Feature_2,Categorical Feature_box,Categorical Feature_fox,Categorical Feature_socks
0,True,False,False,False,False,True
1,False,True,False,False,True,False
2,False,False,True,False,False,True
3,False,True,False,True,False,False
